# Rebuilding T5 Test Set (Patient-wise Split)

This notebook recreates the 5-class Sri Lankan dataset (T5) splits (`train`, `val`, `test`) to ensure that images from the same patient do not leak across different splits. It uses `Patientwise_Data.csv` and `Imagewise_Data.csv` to perform a `GroupShuffleSplit`.

**Source Directory**: `/Users/alokdwivedi/dev/oral-images-dataset/zenodo_10664056_SriLankan`
**Output Directory**: `/Users/alokdwivedi/dev/oral_cavity_5class_dataset_patientwise`
*(Output directory is new so we do not overwrite the existing one used in production until validated)*

In [1]:
import os
import shutil
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit
from pathlib import Path
import json
from tqdm.auto import tqdm

SEED = 42
np.random.seed(SEED)

SOURCE_DIR = "/Users/alokdwivedi/dev/oral-images-dataset/zenodo_10664056_SriLankan"
OUTPUT_DIR = "/Users/alokdwivedi/dev/oral_cavity_5class_dataset_patientwise"

print(f"Source: {SOURCE_DIR}")
print(f"Target: {OUTPUT_DIR}")

Source: /Users/alokdwivedi/dev/oral-images-dataset/zenodo_10664056_SriLankan
Target: /Users/alokdwivedi/dev/oral_cavity_5class_dataset_patientwise


## 1. Load Data

In [7]:
image_df = pd.read_csv(os.path.join(SOURCE_DIR, 'Imagewise_Data.csv'))
patient_df = pd.read_csv(os.path.join(SOURCE_DIR, 'Patientwise_Data.csv'))

# Extract Patient ID from Image Name (e.g., R-01-01 -> R-01)
# The format seems to generally be PREFIX-PATIENTID-IMAGEID
def extract_patient_id(image_name):
    parts = image_name.split('-')
    if len(parts) >= 3:
        # N-256-01 -> N-256
        return f"{parts[0]}-{parts[1]}"
    elif len(parts) == 2:
        # Fallback
        return parts[0]
    return image_name

image_df['Patient ID'] = image_df['Image Name'].apply(extract_patient_id)

print(f"Images: {len(image_df)}")
print(f"Patients in Patientwise_Data: {len(patient_df)}")
print(f"Unique patients in Imagewise_Data: {image_df['Patient ID'].nunique()}")

Images: 3000
Patients in Patientwise_Data: 714
Unique patients in Imagewise_Data: 714


## 2. Check File Existence & Map Classes
The images are stored in `Images/`. They have `.jpg` extension usually.

In [8]:
image_dir = os.path.join(SOURCE_DIR, 'Images')
existing_images = set(os.listdir(image_dir))

def get_filename(base_name):
    # Usually they are .jpg, let's check .jpg and .jpeg
    if f"{base_name}.jpg" in existing_images:
        return f"{base_name}.jpg"
    elif f"{base_name}.jpeg" in existing_images:
        return f"{base_name}.jpeg"
    elif base_name in existing_images:
        return base_name
    # fallback to just returning the .jpg string if missing
    return f"{base_name}.jpg"

image_df['filename'] = image_df['Image Name'].apply(get_filename)

# Map the dataset classes to our 5-class structure:
# Dataset Category -> Project Class
# OCA -> OCA
# OPMD -> OPMD
# Benign -> Benign
# Healthy -> Healthy
# (Unknown will be added later from other sources if needed, this dataset only has the 4 classes)

class_mapping = {
    'OCA': 'OCA',
    'OPMD': 'OPMD',
    'Benign': 'Benign',
    'Healthy': 'Healthy'
}

image_df['target_class'] = image_df['Category'].map(class_mapping)

# Filter to only files that actually exist to be safe
image_df['exists'] = image_df['filename'].apply(lambda x: x in existing_images)
print(f"Images found on disk: {image_df['exists'].sum()} out of {len(image_df)}")
image_df = image_df[image_df['exists']].copy()

Images found on disk: 3000 out of 3000


## 3. Patient-wise Split
We want roughly 70% train, 15% val, 15% test.

In [9]:
# Step 1: Split into Train (70%) and Temp (30%)
gss_train = GroupShuffleSplit(n_splits=1, train_size=0.7, random_state=SEED)
train_idx, temp_idx = next(gss_train.split(image_df, groups=image_df['Patient ID']))

train_df = image_df.iloc[train_idx].copy()
temp_df = image_df.iloc[temp_idx].copy()

# Step 2: Split Temp into Val (50% of Temp = 15% overall) and Test (50% of Temp = 15% overall)
gss_val = GroupShuffleSplit(n_splits=1, train_size=0.5, random_state=SEED)
val_idx, test_idx = next(gss_val.split(temp_df, groups=temp_df['Patient ID']))

val_df = temp_df.iloc[val_idx].copy()
test_df = temp_df.iloc[test_idx].copy()

train_df['split'] = 'train'
val_df['split'] = 'val'
test_df['split'] = 'test'

final_df = pd.concat([train_df, val_df, test_df])

print("Split sizes (images):")
print(final_df['split'].value_counts())
print("\nSplit sizes (patients):")
print(final_df.groupby('split')['Patient ID'].nunique())

Split sizes (images):
split
train    2114
val       458
test      428
Name: count, dtype: int64

Split sizes (patients):
split
test     108
train    499
val      107
Name: Patient ID, dtype: int64


## 4. Copy Files to Output Directory

In [10]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

categories = class_mapping.values()

for split in ['train', 'val', 'test']:
    for cat in categories:
        os.makedirs(os.path.join(OUTPUT_DIR, split, cat), exist_ok=True)

print("Copying files...")
for _, row in tqdm(final_df.iterrows(), total=len(final_df)):
    src_path = os.path.join(image_dir, row['filename'])
    dest_path = os.path.join(OUTPUT_DIR, row['split'], row['target_class'], row['filename'])
    
    # Copy file
    shutil.copy2(src_path, dest_path)

print(f"Dataset successfully created at {OUTPUT_DIR}")

Copying files...


  0%|          | 0/3000 [00:00<?, ?it/s]

Dataset successfully created at /Users/alokdwivedi/dev/oral_cavity_5class_dataset_patientwise


## 5. Save Manifests
The current frozen_test_sets.py expects T5.json to be created from test set. Let's create `T5_patientwise_manifest.json` right away as a reference.

In [11]:
t5_manifest_list = []

for _, row in test_df.iterrows():
    dest_path = os.path.join(OUTPUT_DIR, 'test', row['target_class'], row['filename'])
    t5_manifest_list.append(dest_path)

manifest_path = os.path.join(OUTPUT_DIR, 'T5_patientwise_test_manifest.json')
with open(manifest_path, 'w') as f:
    json.dump(t5_manifest_list, f, indent=2)

print(f"Saved test manifest to {manifest_path} with {len(t5_manifest_list)} images.")

Saved test manifest to /Users/alokdwivedi/dev/oral_cavity_5class_dataset_patientwise/T5_patientwise_test_manifest.json with 428 images.
